# Low Chin Tone Detection from EMG Signal - Version 3

This version loads raw EDF signals directly without normalization and merges continuous events.
- Loads EDF files directly using modules from sleep_stage
- Processes raw EMG signals without normalization
- Divides 30-second epochs into 10 segments of 3 seconds each
- Calculates energy for each segment
- Merges continuous low chin tone events into single events
- Averages RMS values for merged events
- Generates XML output with merged events and averaged RMS values

In [27]:
import sys
import os
sys.path.append('/home/honeynaps/data/shared/integrate/sleep_stage')

import random
import torch
import torch.nn as nn
import numpy as np
import pickle
import natsort
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import timedelta, datetime
import datetime as dt

# Import sleep_stage modules
from sleep_stage.modules.iofiles import edf as edf_io
from sleep_stage.modules.preprocessing import prep_psg_signal, prep_psg_signal_with_missing
from sleep_stage.prep_window_wise import epoching_from_time
from sleep_stage.utils.tools import str2bool

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
torch.manual_seed(5)
np.random.seed(5)
random.seed(5)

In [28]:
def load_edf_raw(edf_path, fs=50):
    """
    Load EDF file and extract raw signals without normalization
    Based on sleep_score.py and SleepFinal.py
    
    Args:
        edf_path: Path to EDF file
        fs: Sampling frequency (default: 50Hz)
    
    Returns:
        sigs: Dictionary of signals
        base_time: Recording start time
        X: Epoched data (n_epochs, n_samples, n_channels)
    """
    # Load EDF using edf_io module
    edf, n_missing_ch = edf_io.load(
        path       = edf_path, 
        preload    = True, 
        resample   = fs, 
        preset     = "STAGENET", 
        exclude    = True,
        missing_ch = 'raise'
    )
    
    base_time = edf.info['meas_date'].replace(tzinfo=None)
    data = edf.get_data()
    
    # Channel mapping (from sleep_score.py)
    SID_MAP = { 
        'F3-':'F3_2', 'F4-':'F4_1', 'C3-':'C3_2', 'C4-':'C4_1', 
        'O1-':'O1_2', 'O2-':'O2_1', 
        'LOC':'LOC' , 'ROC':'ROC', 
        'EMG':'CHIN'
    }
    
    sigs = {}
    for i in range(len(edf.ch_names)):
        name = edf.ch_names[i]
        if name in SID_MAP:
            sigs[SID_MAP[name]] = data[i]
        else:
            sigs[SID_MAP[name[:3]]] = data[i]
    
    # Channel sequence (from SleepFinal.py)
    SID_SEQs = ['F3_2', 'F4_1', 'C3_2', 'C4_1', 'O1_2', 'O2_1', 'LOC', 'ROC', 'CHIN']
    ordered_data = [None for _ in range(len(SID_SEQs))]
    
    for sid, sig in sigs.items():
        if sid in SID_SEQs:
            i = SID_SEQs.index(sid)
            ordered_data[i] = sig
    
    # Check for missing channels
    missing_channels = [i for i in range(9) if ordered_data[i] is None]
    ordered_data = [d for d in ordered_data if d is not None]
    ordered_data = np.array(ordered_data)
    
    # Epoch the data
    X = epoching_from_time(ordered_data.T, base_time, base_time, sfreq=fs)
    
    medians = np.median(ordered_data, axis=1)
    for i in range(ordered_data.shape[0]):
        ordered_data[i] -= medians[i]
        
    return sigs, base_time, X, edf.ch_names

In [29]:
def calculate_emg_energy(emg_signal, segment_length=150):
    """
    Calculate energy for 3-second segments of EMG signal
    
    Args:
        emg_signal: EMG signal (1500 samples for 30 seconds at 50Hz)
        segment_length: Length of each segment (150 samples = 3 seconds at 50Hz)
    
    Returns:
        energy_values: Array of 10 energy values for each 3-second segment
    """
    energy_values = []
    
    for i in range(0, len(emg_signal), segment_length):
        segment = emg_signal[i:i+segment_length]
        if len(segment) == segment_length:
            # Calculate RMS energy
            energy = np.sqrt(np.mean(segment**2))
            energy_values.append(energy)
    
    return np.array(energy_values)

In [30]:
def load_sleep_stage_xml(xml_path):
    """
    Load sleep stage XML and return the first epoch onset time and labels
    
    Args:
        xml_path: Path to the SLEEP XML file
    
    Returns:
        first_epoch_onset: datetime object of the first epoch
        sleep_stages: List of sleep stage labels
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        sleep_stages = []
        first_epoch_onset = None
        
        # Parse all annotations to get sleep stages
        for i, annotation in enumerate(root.findall("annotation")):
            if i == 0:
                onset_text = annotation.find("onset").text
                first_epoch_onset = dt.datetime.strptime(onset_text, "%Y-%m-%dT%H:%M:%S.%f")
            
            description = annotation.find("description").text
            # Convert stage description to numeric label
            if 'W' in description:
                sleep_stages.append(0)
            elif "R" in description:
                sleep_stages.append(1)
            elif "1" in description:
                sleep_stages.append(2)
            elif "2" in description:
                sleep_stages.append(3)
            elif "3" in description:
                sleep_stages.append(4)
        
        return first_epoch_onset, sleep_stages
    except Exception as e:
        print(f"Error reading SLEEP XML {xml_path}: {e}")
        return None, []

In [31]:
def merge_continuous_events(segment_data, threshold, segment_duration=3.0):
    """
    Merge continuous low chin tone segments into single events with averaged RMS
    
    Args:
        segment_data: List of tuples (energy_value, is_low_chin) for each 3-second segment
        threshold: Energy threshold for low chin tone
        segment_duration: Duration of each segment in seconds
    
    Returns:
        merged_events: List of dicts with 'start', 'duration', 'avg_rms', 'type'
    """
    merged_events = []
    current_event = None
    
    for seg_idx, (energy, is_low) in enumerate(segment_data):
        start_time = seg_idx * segment_duration
        
        if is_low:  # Low chin tone segment
            if current_event is None or current_event['type'] != 'LOW':
                # Start new LOW event
                if current_event is not None:
                    # Calculate average RMS for the previous event
                    current_event['avg_rms'] = np.mean(current_event['rms_values'])
                    merged_events.append(current_event)
                
                current_event = {
                    'start': start_time,
                    'duration': segment_duration,
                    'rms_values': [energy],
                    'type': 'LOW'
                }
            else:
                # Continue current LOW event
                current_event['duration'] += segment_duration
                current_event['rms_values'].append(energy)
        else:  # High chin tone segment
            if current_event is None or current_event['type'] != 'HIGH':
                # Start new HIGH event
                if current_event is not None:
                    # Calculate average RMS for the previous event
                    current_event['avg_rms'] = np.mean(current_event['rms_values'])
                    merged_events.append(current_event)
                
                current_event = {
                    'start': start_time,
                    'duration': segment_duration,
                    'rms_values': [energy],
                    'type': 'HIGH'
                }
            else:
                # Continue current HIGH event
                current_event['duration'] += segment_duration
                current_event['rms_values'].append(energy)
    
    # Add the last event
    if current_event is not None:
        current_event['avg_rms'] = np.mean(current_event['rms_values'])
        merged_events.append(current_event)
    
    return merged_events

In [32]:
def save_merged_chin_xml(meas_date, merged_events_list, xml_save_path, min_duration=1.0):
    """
    Save merged chin tone events to XML file with averaged RMS values
    
    Args:
        meas_date: Recording start datetime
        merged_events_list: List of merged events with averaged RMS
        xml_save_path: Path to save XML file
        min_duration: Minimum duration for events in seconds
    """
    root = ET.Element("annotationlist")
    
    # Calculate total recording duration
    if merged_events_list:
        last_event = merged_events_list[-1]
        recording_duration = last_event['start'] + last_event['duration']
    else:
        recording_duration = 0
    
    ET.SubElement(root, "recording_duration").text = f"{recording_duration:.6f}"
    
    event_count = 0
    for event in merged_events_list:
        # Only include events longer than minimum duration
        if event['duration'] < min_duration:
            continue
        
        # Calculate onset time
        onset_time = meas_date + timedelta(seconds=event['start'])
        
        # Create description with TYPE_AVG_RMS format
        description = f"{event['type']}_CHIN_TONE_{event['avg_rms']:.4f}"
        
        annotation = ET.SubElement(root, "annotation")
        
        onset_elem = ET.SubElement(annotation, "onset")
        onset_elem.text = onset_time.strftime("%Y-%m-%dT%H:%M:%S.%f")
        
        duration_elem = ET.SubElement(annotation, "duration")
        duration_elem.text = f"{event['duration']:.6f}"
        
        desc_elem = ET.SubElement(annotation, "description")
        desc_elem.text = description
        
        location_elem = ET.SubElement(annotation, "location")
        location_elem.text = "EEG-EMG"
        
        event_count += 1
    
    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ", level=0)
    
    with open(xml_save_path, "wb") as fp:
        fp.write(b'<?xml version="1.0" encoding="UTF-8"?>\n')
        tree.write(fp, encoding="UTF-8", xml_declaration=False)
    
    return event_count

In [33]:
# Configuration
fs = 50
emg_channel_idx = 8  # EMG is the 9th channel (0-indexed)
segment_length = 50  # 3 seconds at 50Hz
epoch_length = 1500   # 30 seconds at 50Hz

# Data paths
edf_dir = '/home/honeynaps/data/GOLDEN/EDF2'
sleep_xml_dir = '/home/honeynaps/data/GOLDEN/EBX2/SLEEP'
output_dir = '/home/honeynaps/data/shared/integrate/output'

# Get list of EDF files
edf_files = natsort.natsorted([f for f in os.listdir(edf_dir) if f.endswith('.edf')])
print(f"Found {len(edf_files)} EDF files")

# Process first 5 files
max_files = 5
edf_files = edf_files[:max_files]
print(f"Processing {len(edf_files)} files")

Found 16 EDF files
Processing 5 files


In [34]:
# Load data and extract EMG signals
all_emg_data = []
all_energy_values = []
all_sleep_stages = []
rem_energy_values = []
file_data_mapping = []
sleep_stage_names = ['Wake', 'REM', 'N1', 'N2', 'N3']

print("Loading EDF files and calculating EMG energy...")

for file_idx, edf_file in enumerate(edf_files):
    print(f"\nProcessing file {file_idx + 1}/{len(edf_files)}: {edf_file}")
    
    edf_path = os.path.join(edf_dir, edf_file)
    base_name = edf_file.replace('.edf', '')
    
    # Load corresponding SLEEP XML to get sleep stages
    sleep_xml_path = os.path.join(sleep_xml_dir, f"{base_name}_SLEEP.xml")
    
    if not os.path.exists(sleep_xml_path):
        print(f"  Warning: SLEEP XML not found: {sleep_xml_path}")
        continue
    
    # Load sleep stages from XML
    first_onset, sleep_stages = load_sleep_stage_xml(sleep_xml_path)
    
    if first_onset is None:
        print(f"  Error loading SLEEP XML")
        continue
    
    # Load raw EDF data
    try:
        sigs, base_time, X, ch_names = load_edf_raw(edf_path, fs=fs)
        print(f"  Loaded EDF: {X.shape[0]} epochs, {X.shape[1]} samples, {X.shape[2]} channels")
        print(f"  Base time: {base_time}")
    except Exception as e:
        print(f"  Error loading EDF: {e}")
        continue
    
    # Process each epoch
    n_epochs = min(X.shape[0], len(sleep_stages))
    print(f"  Processing {n_epochs} epochs")
    
    for epoch_idx in range(n_epochs):
        # Extract EMG signal (last channel)
        emg_signal = X[epoch_idx, :, emg_channel_idx]
        emg_signal *= 1000  # Scale to microvolts
        sleep_stage = sleep_stages[epoch_idx]
        
        # Calculate energy for 10 segments of 3 seconds each
        energy_values = calculate_emg_energy(emg_signal, segment_length)
        
        if len(energy_values) == 10:  # Ensure we have exactly 10 segments
            all_emg_data.append(emg_signal)
            all_energy_values.extend(energy_values)
            all_sleep_stages.extend([sleep_stage] * 10)
            file_data_mapping.append({
                'file_idx': file_idx, 
                'file_name': edf_file, 
                'base_name': base_name,
                'local_epoch_idx': epoch_idx,
                'base_time': base_time  # Use base_time from EDF
            })
            
            # Collect REM energy values for threshold calculation
            if sleep_stage == 1:  # REM stage
                rem_energy_values.extend(energy_values)

all_emg_data = np.array(all_emg_data)
all_energy_values = np.array(all_energy_values)
all_sleep_stages = np.array(all_sleep_stages)
rem_energy_values = np.array(rem_energy_values)

print(f"\nTotal epochs processed: {len(all_emg_data)}")
print(f"Total 3-second segments: {len(all_energy_values)}")
print(f"REM segments for threshold calculation: {len(rem_energy_values)}")

Loading EDF files and calculating EMG energy...

Processing file 1/5: SCH-230114R3_M-60-OV-SE.edf
  Loaded EDF: 784 epochs, 1500 samples, 9 channels
  Base time: 2023-01-14 20:58:49
  Processing 779 epochs

Processing file 2/5: SCH_F_20_OB_231128R4_NO.edf
  Loaded EDF: 920 epochs, 1500 samples, 9 channels
  Base time: 2023-11-28 22:11:30
  Processing 894 epochs

Processing file 3/5: SCH_F_20_OV_230715R3_MO.edf
  Loaded EDF: 898 epochs, 1500 samples, 9 channels
  Base time: 2023-07-15 22:12:17
  Processing 890 epochs

Processing file 4/5: SCH_F_40_NW_230511R3_SE.edf
  Loaded EDF: 834 epochs, 1500 samples, 9 channels
  Base time: 2023-05-11 22:34:32
  Processing 824 epochs

Processing file 5/5: SCH_F_40_NW_231130R4_MO.edf
  Loaded EDF: 848 epochs, 1500 samples, 9 channels
  Base time: 2023-11-30 22:05:57
  Processing 836 epochs

Total epochs processed: 0
Total 3-second segments: 0
REM segments for threshold calculation: 0


In [35]:
# Calculate threshold based on REM statistics
threshold_percentile = 75  # More relaxed threshold

if len(rem_energy_values) > 0:
    threshold = np.percentile(rem_energy_values, threshold_percentile)
else:
    threshold = 0.5  # Default threshold if no REM data

print(f"Low chin tone threshold: {threshold:.4f} (using {threshold_percentile}th percentile)")

# Basic statistics
if len(rem_energy_values) > 0:
    print(f"\nREM energy statistics:")
    print(f"  Mean: {np.mean(rem_energy_values):.4f}")
    print(f"  Std: {np.std(rem_energy_values):.4f}")
    print(f"  25th percentile: {np.percentile(rem_energy_values, 25):.4f}")
    print(f"  50th percentile: {np.percentile(rem_energy_values, 50):.4f}")
    print(f"  75th percentile: {np.percentile(rem_energy_values, 75):.4f}")
    print(f"\nNote: Using {threshold_percentile}th percentile for more relaxed low chin tone detection")

Low chin tone threshold: 0.5000 (using 75th percentile)


In [36]:
# Statistical Analysis and Visualization
if len(all_energy_values) > 0:
    # Create DataFrame for analysis
    df = pd.DataFrame({
        'energy': all_energy_values,
        'sleep_stage': all_sleep_stages,
        'stage_name': [sleep_stage_names[int(stage)] for stage in all_sleep_stages]
    })

    # Overall energy distribution
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 1. Overall energy histogram
    axes[0, 0].hist(all_energy_values, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.3f}')
    axes[0, 0].set_xlabel('EMG Energy (RMS)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Overall EMG Energy Distribution (Raw Signal)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Energy distribution by sleep stage
    stage_colors = ['red', 'blue', 'green', 'orange', 'purple']
    for stage in range(5):
        stage_data = df[df['sleep_stage'] == stage]['energy']
        if len(stage_data) > 0:
            axes[0, 1].hist(stage_data, bins=30, alpha=0.6, 
                           label=f'{sleep_stage_names[stage]} (n={len(stage_data)})',
                           color=stage_colors[stage])

    axes[0, 1].axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.3f}')
    axes[0, 1].set_xlabel('EMG Energy (RMS)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('EMG Energy Distribution by Sleep Stage (Raw Signal)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Box plot by sleep stage
    sns.boxplot(data=df, x='stage_name', y='energy', ax=axes[1, 0])
    axes[1, 0].axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.3f}')
    axes[1, 0].set_xlabel('Sleep Stage')
    axes[1, 0].set_ylabel('EMG Energy (RMS)')
    axes[1, 0].set_title('EMG Energy by Sleep Stage - Box Plot (Raw Signal)')
    axes[1, 0].legend()
    axes[1, 0].tick_params(axis='x', rotation=45)

    # 4. Statistical summary table
    stats_summary = df.groupby('stage_name')['energy'].agg(['count', 'mean', 'std', 'min', 'max']).round(4)
    axes[1, 1].axis('tight')
    axes[1, 1].axis('off')
    table = axes[1, 1].table(cellText=stats_summary.values, 
                            rowLabels=stats_summary.index,
                            colLabels=stats_summary.columns,
                            cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.5)
    axes[1, 1].set_title('Statistical Summary by Sleep Stage')

    plt.tight_layout()
    plt.show()

    # Print summary statistics
    print("\nDetailed Statistics by Sleep Stage:")
    print(stats_summary)

In [37]:
# Generate XML output with merged events for each file
print("\n" + "="*60)
print("GENERATING XML OUTPUT WITH MERGED EVENTS")
print("="*60)

os.makedirs(output_dir, exist_ok=True)

# Group data by file
file_groups = {}
for mapping in file_data_mapping:
    file_idx = mapping['file_idx']
    if file_idx not in file_groups:
        file_groups[file_idx] = {
            'file_name': mapping['file_name'],
            'base_name': mapping['base_name'],
            'base_time': mapping['base_time'],  # Use base_time from EDF
            'epochs': []
        }
    file_groups[file_idx]['epochs'].append(mapping['local_epoch_idx'])

# Process each file
file_results = {}

for file_idx, file_info in file_groups.items():
    file_name = file_info['file_name']
    base_name = file_info['base_name']
    base_time = file_info['base_time']  # Use base_time from EDF
    
    print(f"\nProcessing: {file_name}")
    print(f"  Base time (EDF start): {base_time}")
    
    # Collect EMG data and create segment data for this file
    file_segment_data = []
    
    for i, mapping in enumerate(file_data_mapping):
        if mapping['file_idx'] == file_idx:
            emg_signal = all_emg_data[i]
            energy_values = calculate_emg_energy(emg_signal, segment_length)
            
            # Create segment data with energy and classification
            for energy in energy_values:
                is_low = energy < threshold
                file_segment_data.append((energy, is_low))
    
    if len(file_segment_data) == 0:
        continue
    
    print(f"  Number of 3-second segments: {len(file_segment_data)}")
    
    # Merge continuous events
    merged_events = merge_continuous_events(file_segment_data, threshold, segment_duration=3.0)
    
    print(f"  Merged into {len(merged_events)} events")
    
    # Count LOW and HIGH events
    low_events = [e for e in merged_events if e['type'] == 'LOW']
    high_events = [e for e in merged_events if e['type'] == 'HIGH']
    
    print(f"  LOW chin tone events: {len(low_events)}")
    print(f"  HIGH chin tone events: {len(high_events)}")
    
    # Calculate total durations
    low_duration = sum(e['duration'] for e in low_events)
    high_duration = sum(e['duration'] for e in high_events)
    total_duration = low_duration + high_duration
    
    print(f"  LOW chin tone duration: {low_duration:.1f}s ({low_duration/total_duration*100:.1f}%)")
    print(f"  HIGH chin tone duration: {high_duration:.1f}s ({high_duration/total_duration*100:.1f}%)")
    
    # Save merged events XML
    xml_filename = f"{base_name}_CHIN_MERGED.xml"
    xml_save_path = os.path.join(output_dir, xml_filename)
    
    event_count = save_merged_chin_xml(
        meas_date=base_time,  # Use base_time from EDF
        merged_events_list=merged_events,
        xml_save_path=xml_save_path,
        min_duration=1.0  # Minimum 1 second events
    )
    
    print(f"  XML saved: {xml_filename}")
    print(f"  Events saved (≥1s): {event_count}")
    
    # Store results
    file_results[file_name] = {
        'xml_path': xml_save_path,
        'total_events': len(merged_events),
        'saved_events': event_count,
        'low_events': len(low_events),
        'high_events': len(high_events),
        'low_duration': low_duration,
        'high_duration': high_duration,
        'base_time': base_time
    }

print("\n" + "="*60)
print(f"XML files saved to: {output_dir}")
print(f"Threshold used: {threshold:.4f}")
print("="*60)


GENERATING XML OUTPUT WITH MERGED EVENTS

XML files saved to: /home/honeynaps/data/shared/integrate/output
Threshold used: 0.5000


In [38]:
# Display example of merged events
if file_results:
    print("\n" + "="*60)
    print("EXAMPLE OF MERGED EVENTS")
    print("="*60)
    
    # Get first file's merged events for example
    first_file_idx = list(file_groups.keys())[0]
    first_file_info = file_groups[first_file_idx]
    
    # Recreate segment data for first file
    example_segment_data = []
    for i, mapping in enumerate(file_data_mapping[:50]):  # First 50 epochs for example
        if mapping['file_idx'] == first_file_idx:
            emg_signal = all_emg_data[i]
            energy_values = calculate_emg_energy(emg_signal, segment_length)
            for energy in energy_values:
                is_low = energy < threshold
                example_segment_data.append((energy, is_low))
    
    if example_segment_data:
        # Get first 10 merged events
        example_merged = merge_continuous_events(example_segment_data[:100], threshold, segment_duration=3.0)
        
        print(f"\nFirst 10 merged events from {first_file_info['file_name']}:")
        print(f"{'Event':<8} {'Type':<6} {'Start(s)':<10} {'Duration(s)':<12} {'Avg RMS':<10} {'# Segments':<10}")
        print("-" * 70)
        
        for i, event in enumerate(example_merged[:10]):
            n_segments = int(event['duration'] / 3.0)
            print(f"{i+1:<8} {event['type']:<6} {event['start']:<10.1f} {event['duration']:<12.1f} "
                  f"{event['avg_rms']:<10.4f} {n_segments:<10}")
        
        print("\nNote: Each merged event combines consecutive segments of the same type")
        print("      RMS values are averaged across all segments in the event")

In [39]:
# Final Summary
print("\n" + "="*60)
print("FINAL ANALYSIS SUMMARY - Version 3 (Merged Events)")
print("="*60)

print(f"\nData Loading Method:")
print(f"  - Direct EDF loading using sleep_stage modules")
print(f"  - Raw signal processing without normalization")
print(f"  - Using base_time from EDF file")
print(f"  - Channel mapping based on sleep_score.py")

print(f"\nDataset Information:")
print(f"  Total files processed: {len(file_groups)}")
print(f"  Total 30-second epochs: {len(all_emg_data)}")
print(f"  Total 3-second segments: {len(all_energy_values)}")

if len(rem_energy_values) > 0:
    print(f"\nThreshold Calculation:")
    print(f"  Based on REM stage statistics")
    print(f"  REM segments used: {len(rem_energy_values)}")
    print(f"  Threshold value: {threshold:.4f}")
    print(f"  Method: {threshold_percentile}th percentile of REM energy values")

# Classification results
if len(all_energy_values) > 0:
    low_chin_classifications = all_energy_values < threshold
    low_chin_count = np.sum(low_chin_classifications)
    low_chin_percentage = (low_chin_count / len(all_energy_values)) * 100
    
    print(f"\nLow Chin Tone Detection Results:")
    print(f"  Total segments analyzed: {len(all_energy_values)}")
    print(f"  Low chin tone detected: {low_chin_count} ({low_chin_percentage:.1f}%)")
    print(f"  Normal chin tone: {len(all_energy_values) - low_chin_count} ({100 - low_chin_percentage:.1f}%)")
    
    # Breakdown by sleep stage
    print(f"\nLow Chin Tone by Sleep Stage:")
    for stage in range(5):
        stage_mask = all_sleep_stages == stage
        if np.any(stage_mask):
            stage_total = np.sum(stage_mask)
            stage_low_chin = np.sum(low_chin_classifications[stage_mask])
            stage_percentage = (stage_low_chin / stage_total) * 100 if stage_total > 0 else 0
            print(f"  {sleep_stage_names[stage]:>5}: {stage_low_chin:>4}/{stage_total:<4} ({stage_percentage:>5.1f}%)")

print(f"\nMerged Events Summary:")
if file_results:
    total_low_events = sum(r['low_events'] for r in file_results.values())
    total_high_events = sum(r['high_events'] for r in file_results.values())
    total_saved_events = sum(r['saved_events'] for r in file_results.values())
    
    print(f"  Total merged events: {total_low_events + total_high_events}")
    print(f"  LOW chin tone events: {total_low_events}")
    print(f"  HIGH chin tone events: {total_high_events}")
    print(f"  Events saved (≥1s): {total_saved_events}")

print(f"\nXML Output Summary:")
print(f"  Output directory: {output_dir}")
print(f"  XML file format: [filename]_CHIN_MERGED.xml")
print(f"  Description format: TYPE_CHIN_TONE_AVG_RMS")
print(f"  Example: LOW_CHIN_TONE_0.0025 or HIGH_CHIN_TONE_0.0078")
print(f"  Continuous segments merged into single events")
print(f"  RMS values averaged across merged segments")

print("\n" + "="*60)
print("Analysis completed successfully!")
print("Version 3: Raw EDF with merged continuous events")
print("="*60)


FINAL ANALYSIS SUMMARY - Version 3 (Merged Events)

Data Loading Method:
  - Direct EDF loading using sleep_stage modules
  - Raw signal processing without normalization
  - Using base_time from EDF file
  - Channel mapping based on sleep_score.py

Dataset Information:
  Total files processed: 0
  Total 30-second epochs: 0
  Total 3-second segments: 0

Merged Events Summary:

XML Output Summary:
  Output directory: /home/honeynaps/data/shared/integrate/output
  XML file format: [filename]_CHIN_MERGED.xml
  Description format: TYPE_CHIN_TONE_AVG_RMS
  Example: LOW_CHIN_TONE_0.0025 or HIGH_CHIN_TONE_0.0078
  Continuous segments merged into single events
  RMS values averaged across merged segments

Analysis completed successfully!
Version 3: Raw EDF with merged continuous events
